# Phase 3b: embed the selected names with LaBSE

Runs on a Colab GPU session. The job is **resumable**: re-running the last cell after a disconnect picks up at the first missing shard.

## Order of operations, and why

1. **Mount Drive.** Shards and the plan go to Drive, not `/content`, so they survive the session ending. This is the whole reason the job is resumable.
2. **Build the plan (CPU).** Decide the rows and write them out. Cheap, inspectable, and it happens before any GPU time is spent.
3. **Calibrate (GPU, minutes).** Sweep batch sizes, keep the fastest that fits in VRAM, then time 50k rows at the winner.
4. **Decide.** Extrapolate the measured rows/sec to the real row count and compare against a session budget. **If it does not fit, this cell says so before anything is written.** That is the cell to read.
5. **Run.** Shard by shard, stopping in time to exit cleanly, skipping shards already on disk.
6. **Verify.** Row count *and* entity ids, because a bad resume can match on totals while duplicating ids.

## The batch size is measured, not chosen

Everything downstream -- shard size, sessions, whether the run is feasible at all -- comes from the 50k timing in cell 4. Reusing 50k because that is what validation used would be the wrong number twice over: 50k is a *sample* for measuring a rate, not a batch size for a 3.4M-row job. LaBSE at 768 dims and 12 layers will not fit in memory at any batch size that large.

## Before the first full run

```
STRICT = False          # the Phase 3b decision: broad, includes accented Latin
SESSION_BUDGET_HOURS    # read cell 4's DECISION line before raising it
```

If cell 4 prints `infeasible`, stop and pick one deliberately: a larger batch size, a smaller encoder, or the strict filter. Do not discover this at hour three.

In [ ]:
# !pip install -q sentence-transformers pyarrow

import os, sys, time, json, shutil
from pathlib import Path

from google.colab import drive
drive.mount('/content/drive')

REPO = Path('/content/drive/MyDrive/business-entity-resolution')
if not REPO.exists():
    raise SystemExit(f'Clone the repo to {REPO} first, then re-run this cell.')
sys.path.insert(0, str(REPO))
os.chdir(REPO)

import pandas as pd
import torch
from src.blocking import embed_runtime as rt
from src.blocking.embed_names import DEFAULT_MODEL, expected_embed_rows

print('torch     ', torch.__version__)
print('cuda      ', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')
if torch.cuda.is_available():
    print('vram GiB  ', round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1))
else:
    raise SystemExit('No GPU. This notebook needs a Colab GPU runtime.')
print('model     ', DEFAULT_MODEL)

## Configuration

`SESSION_BUDGET_HOURS` is deliberately well under Colab's maximum session length. An idle disconnect or a reconnection cycle can eat into the session at any time, and `plan_sessions` applies a further 15% headroom on top. Planning to the exact limit is how the last shard never lands.

`SHARD_ROWS` at 100k is ~195 MiB of fp16 LaBSE vectors per shard: comfortable to write, and 34 shards to cover the test split. See `test_a_100k_shard_of_labse_vectors_is_about_195_mib`.

In [ ]:
STRICT = False                 # the decision from the 3.4M-vs-2.6M review
SPLITS = ['test']              # ['test'] or ['test', 'train']

MODEL = DEFAULT_MODEL          # sentence-transformers/LaBSE
SHARD_ROWS = 100_000
CHUNKSIZE = 250_000            # rows per CSV chunk when building the plan
VRAM_BUDGET_GIB = 12.0         # leave headroom on a 15.8 GiB T4

SESSION_BUDGET_HOURS = 5.0     # usable wall clock for THIS session
STOP_AT_FRACTION = 0.92        # stop between shards with 8% of budget unused
BATCH_CANDIDATES = [64, 128, 256, 512]
TIMED_ROWS = 50_000            # the timed run the plan is extrapolated from

WORK = Path('/content/drive/MyDrive/ber_embeddings')
PLAN_DIR = WORK / ('plan_strict' if STRICT else 'plan_broad')
OUT_DIR = WORK / ('out_strict' if STRICT else 'out_broad')
CALIB_DIR = WORK / 'calibration'
for d in (WORK, CALIB_DIR):
    d.mkdir(parents=True, exist_ok=True)

SOURCE_PATHS = {
    f'{split}_source{i}.tsv': REPO / 'dataset' / split / f'{split}_source{i}.tsv'
    for split in SPLITS for i in (1, 2, 3)
}
missing = [str(p) for p in SOURCE_PATHS.values() if not p.exists()]
assert not missing, f'missing source files: {missing}'
print(f'plan -> {PLAN_DIR}')
print(f'out  -> {OUT_DIR}')

## Cell 3: build the plan (CPU, no model)

Streams each source file, applies the flags, and writes the rows to embed as `plan/shard_NNNNN.parquet` on Drive. Reads only `entity_id` and `business_name` -- the address is not needed to decide which *names* to embed.

Deterministic, so re-running produces the same shard N and a resumed session cannot embed the wrong rows into it.

In [ ]:
t0 = time.time()
plan_summary = rt.build_plan_shards(
    SOURCE_PATHS, PLAN_DIR,
    strict=STRICT, shard_rows=SHARD_ROWS, chunksize=CHUNKSIZE,
)
print(f'\nbuilt in {time.time() - t0:.0f}s')

expected_parts = {n: expected_embed_rows(Path(n).name, strict=STRICT) for n in SOURCE_PATHS}
unknown = [n for n, v in expected_parts.items() if v is None]
assert not unknown, (
    f'no published row expectation for {unknown}, so the plan cannot be '
    'checked against anything. Build the plan and read the count off it.'
)
expected = sum(expected_parts.values())
actual = plan_summary['total_rows']
print(f'plan rows      {actual:,}')
print(f'expected rows  {expected:,}')
print(f'delta          {actual - expected:+,} ({100 * (actual - expected) / expected:+.2f}%)')

TOLERANCE = 0.02   # the published counts are exact; drift means the flags moved
drift = abs(actual - expected) / expected
assert drift < TOLERANCE, (
    f'plan is {100 * (actual - expected) / expected:+.2f}% off the published '
    'expectation; check the flags before spending GPU hours'
)
print(f'\nOK - within {100 * TOLERANCE:.0f}% of the published expectation')

# What makes a resume safe. Two runs of this job differ in model, strict flag
# and shard layout, and each of those changes what the vectors *are* while
# leaving the row counts identical. Hashing them means a shard from the wrong
# run cannot be mistaken for a finished one.
IDENTITY = {
    'model': MODEL,
    'strict': STRICT,
    'shard_rows': SHARD_ROWS,
    'plan_total_rows': actual,
    'splits': SPLITS,
    'storage_dtype': str(rt.STORAGE_DTYPE),
}
print('identity', rt.run_fingerprint(IDENTITY))

## Cell 4: calibrate, then decide

Sweeps batch sizes on a small trial, keeps the fastest that fits in VRAM, then times 50k rows at that batch size. The 50k rate is the only number the plan is extrapolated from.

**Read the `DECISION` line.**

| decision | what to do |
|---|---|
| `single-session` | run cell 5 as is |
| `split` | run cell 5, re-run it after each disconnect until it reports complete |
| `checkpoint-resume` | same, expect several sessions; the shards on Drive are the progress |
| `infeasible` | **stop.** Raise `SESSION_BUDGET_HOURS`, widen `BATCH_CANDIDATES`, switch to a smaller encoder, or switch `STRICT = True`. Do not start and find out later |

In [ ]:
plan_shards = rt.iter_plan_shards(PLAN_DIR)

# Drawn across shards, not from shard 0. Shard 0 is entirely source1, so
# timing it would measure one country's name-length distribution and call it
# the rate for all three. It also decouples TIMED_ROWS from SHARD_ROWS.
calib_texts: list[str] = []
for _shard_id, frame in plan_shards:
    calib_texts.extend(frame['business_name_clean'].fillna('').astype(str).tolist())
    if len(calib_texts) >= TIMED_ROWS:
        break
calib_texts = calib_texts[:TIMED_ROWS]
if len(calib_texts) < TIMED_ROWS:
    raise SystemExit(
        f'plan holds {len(calib_texts):,} rows, need {TIMED_ROWS:,} to time. '
        'Lower TIMED_ROWS.'
    )
print(f'timing on {len(calib_texts):,} names drawn from the first '
      f'{TIMED_ROWS // SHARD_ROWS + 1} plan shard(s)')

print('sweeping batch sizes...')
choices = rt.sweep_batch_sizes(
    calib_texts, rt.load_encoder, BATCH_CANDIDATES,
    model_name=MODEL, rows_per_trial=4_096, vram_budget_gib=VRAM_BUDGET_GIB,
)
for c in choices:
    vram = 'n/a' if c.peak_vram_gib is None else f'{c.peak_vram_gib:.1f} GiB'
    print(f'  batch {c.batch_size:>4}  {c.rows_per_sec:>9,.0f} rows/s  {vram:>10}  '
          f'{"fits" if c.fits else "SKIP: " + c.reason}')

winner = rt.choose_batch_size(choices)
print(f'\nchosen batch size: {winner.batch_size}')

# The authoritative timing: 50k rows at the chosen batch size.
del choices  # free the models the sweep loaded before loading one properly
import torch, gc; gc.collect(); torch.cuda.empty_cache()
model, device = rt.load_encoder(MODEL)
print(f'loaded {MODEL} on {device}')
encode = rt.make_encode_fn(model, winner.batch_size)
throughput = rt.measure_throughput(
    calib_texts[:TIMED_ROWS], encode, batch_size=winner.batch_size
)
print(f'\ntimed {throughput.rows_timed:,} rows at batch {throughput.batch_size}: '
      f'{throughput.rows_per_sec:,.0f} rows/s in {throughput.seconds_timed:.1f}s')

plan = rt.plan_sessions(
    plan_summary['total_rows'], throughput,
    budget_seconds=SESSION_BUDGET_HOURS * 3600, shard_rows=SHARD_ROWS,
)
print()
print(rt.describe_plan(plan))

rt.write_manifest(CALIB_DIR, {
    'model': MODEL, 'device': device, 'strict': STRICT,
    'rows_timed': throughput.rows_timed, 'rows_per_sec': throughput.rows_per_sec,
    'batch_size': throughput.batch_size, 'peak_vram_gib': throughput.peak_vram_gib,
    'seconds_needed': plan.seconds_needed, 'decision': plan.decision,
    'sessions': plan.sessions, 'gpu': torch.cuda.get_device_name(0),
})

if plan.decision == 'infeasible':
    raise SystemExit(
        f'INFEASIBLE: {plan.seconds_needed / 3600:.1f} h of encoding against a '
        f'{SESSION_BUDGET_HOURS} h budget ({plan.sessions} sessions). '
        'Decide deliberately: bigger batch, smaller encoder, or STRICT=True.'
    )

## Cell 5: run

Skips any shard already complete on Drive, writes the rest, and stops between shards once `STOP_AT_FRACTION` of the budget is used -- so it exits cleanly rather than being killed mid-write.

**After a disconnect, just re-run this cell.** It resumes at the first missing shard.

In [ ]:
encode = rt.make_encode_fn(model, plan.batch_size)
t0 = time.time()
outcome = rt.run_shards(
    rt.iter_plan_shards(PLAN_DIR), OUT_DIR, encode,
    max_seconds=SESSION_BUDGET_HOURS * 3600,
    stop_at_fraction=STOP_AT_FRACTION,
    identity=IDENTITY,
)
elapsed = time.time() - t0

print(f'\nsession used {elapsed / 60:.1f} min of {SESSION_BUDGET_HOURS * 60:.0f}')
print(f'  shards written  {outcome.shards_written}')
print(f'  shards skipped  {outcome.shards_skipped}  (already on Drive)')
print(f'  rows            {outcome.rows_written:,}')
if outcome.shards_remaining:
    print(f'  REMAINING       {len(outcome.shards_remaining)} shards: '
          f'{outcome.shards_remaining[:5]}{"..." if len(outcome.shards_remaining) > 5 else ""}')
    print(f'  reason          {outcome.reason}')
    print(f'\nRe-run this cell in a new session to continue from shard '
          f'{outcome.shards_remaining[0]}.')
else:
    print('\nAll shards written. Run the verification cell.')

rt.write_manifest(OUT_DIR, {
    'fingerprint': rt.run_fingerprint(IDENTITY),
    'identity': IDENTITY,
    'batch_size': plan.batch_size,
    'rows_per_sec_measured': throughput.rows_per_sec,
    'session_minutes': round(elapsed / 60, 1),
    'shards_written_this_session': outcome.shards_written,
    'shards_remaining': len(outcome.shards_remaining),
})

## Cell 6: verify

Checks entity ids, not just row counts. Two sessions that wrote overlapping ranges would double-count, and a stale shard from another run would leave the total looking plausible while the ids were wrong -- which is the failure that survives to the evaluation and shows up as a recall bug nobody can trace.

In [ ]:
import pandas as pd

expected_ids = set(
    pd.concat([f for _i, f in rt.iter_plan_shards(PLAN_DIR)])['entity_id'].astype(str)
)

report = rt.verify_shards(
    OUT_DIR,
    expected_total_rows=plan_summary['total_rows'],
    expected_ids=expected_ids,
    identity=IDENTITY,
)
for k, v in report.items():
    if isinstance(v, list):
        continue
    print(f'  {k:24} {v:,}' if isinstance(v, int) else f'  {k:24} {v}')
if report.get('stale_shards'):
    print('\n  stale shards:', report['stale_shard_names'])
    print('  these were written by a different run -- see identity above')

assert report['ok'], report
print('\nOK - every planned row is embedded exactly once')

print(f'\nembeddings: {OUT_DIR}')
print(f'manifest:   {rt.read_manifest(OUT_DIR)}')

## Next: the ANN index

Not built here, and it is the next real decision. At the measured 3,397,040 planned rows, 768-d vectors are 4.9 GiB as fp16 and 9.7 GiB as fp32 -- neither fits in RAM alongside anything else, so the index has to be either memory-mapped on disk (`faiss.OnDiskInverted`) or built per source file and searched as a union. Both are defensible; the choice depends on the candidate-set sizes `generate_candidates` will need, which is not written yet.

Worth measuring before committing: what fraction of recall the embedding side is actually contributing, on `train_ground_truth`. If the accent-only rows that the strict filter drops turn out to contribute nothing, that is the evidence to narrow the filter with -- measured, not assumed.